# Compare spin simulations and finite-difference trajectories
Run these cells in order from the project directory. Both existing programs run sequentially for every physical parameter set in `config/combined_parameters.txt`. Lists form a **Cartesian product**, not matched pairs. Cases sharing all physical parameters except m0 share one figure: its magnetisation and energy panels each overlay all simulation paths and finite-difference curves for every m0.

These are a finite-N stochastic trajectory and an approximate replica-symmetric density solution. Agreement is not guaranteed; no replica q is measured by the simulation. Simulation time is attempted updates per spin, matching the continuous-time heat-bath rate convention. Finite-N initialization rounds the requested m0 to an allowed population; the FD initial density is discretized and normalized on its grid. Energy uses the same observable in both programs, but directed interactions are not generated by this energy alone.

Install notebook dependencies with `python -m pip install -r requirements-notebook.txt`.


`asymmetry` is k from thesis equation (1.18), with **0 <= k <= 1** in both methods: 0 is symmetric, 1 is fully asymmetric. It is not the reciprocal-coupling correlation, which is (1-k²)/(1+k²).
Set `n_runs` in `config/combined_parameters.txt` to overlay multiple simulated paths per physical set, sharing fixed couplings, against one FD curve. All paths are saved in the comparison PNG. The workload warning counts physical sets × n_runs.


In [ ]:
from pathlib import Path
import sys

# Run this notebook from the project root.
python_dir = Path("python").resolve()
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))
import importlib
import combined_runs
importlib.reload(combined_runs)
from combined_runs import read_combined_config, print_preview, run_sweep, load_manifest, plot_comparisons

parameter_file = Path("config/combined_parameters.txt")
settings = read_combined_config(parameter_file)
print_preview(settings)

## Run both methods
This cell starts new calculations and creates a timestamped sweep folder. **Physical sets × `n_runs` above `warning_threshold` (default 10)** triggers a warning and an input prompt. Type exactly `RUN` to continue; any other response cancels without starting calculations. The preview also shows solver count, duration and simulation matrix memory; even a single high-resolution run can be expensive.

Progress streams below. Each run has its own parameter file, log and standard NPZ archive. A failed solver is recorded and the remaining runs continue; failure is never treated as a successful comparison. Interrupting the cell terminates the active child process and retains a manifest of completed work. Rerunning this cell starts a fresh sweep. Numerical tolerances are never automatically relaxed.

In [ ]:
# Re-read the file here so edits since the preview are validated before running.
settings = read_combined_config(parameter_file)
print_preview(settings)
manifest_path = None  # Clear a previous result if this run is cancelled or interrupted.
manifest_path = run_sweep(settings, parameter_file)

## Review outcomes and plot
Leave `saved_manifest` unset to inspect the sweep above. To reopen results without rerunning either solver, set it to a saved `manifest.json` and run the import cell followed by these cells. Failed methods with saved partial trajectories are plotted up to their last completed step and labelled as partial. Failed or interrupted methods without saved data are labelled without a curve. Legends identify m0. Each panel uses each method's own saved time samples, without interpolation or forced matching.

The same seed is reused across parameter sets. Coupling disorder is therefore shared when only temperature, j0 or m0 varies (as in the C parameter sweeps); transformed disorder shares underlying normal draws when asymmetry or j1 varies. These are correlated comparisons, not independent disorder samples or ensemble averages.
Figures are also saved as PNG files in the project-level `figures/<sweep_folder_name>/` folder, alongside `Output` rather than inside it. Replotting replaces the corresponding case figures.

Energy simulation paths use distinct green shades for different m0 values, identified in the energy legend. Finite-difference curves remain dashed orange.

For k=0, black dotted horizontal lines show the RS equilibrium magnetisation and energy on the positive magnetisation branch, independently of m0. Repeated reference values are drawn once and labelled as the positive branch. No equilibrium reference is applied for k>0.


In [ ]:
saved_manifest = None  # e.g. Path("Output/comparison_.../manifest.json")
selected_manifest = saved_manifest if saved_manifest is not None else globals().get("manifest_path")
if selected_manifest is None:
    manifest = None
    print("No completed sweep selected. Run the sweep or set saved_manifest.")
else:
    manifest = load_manifest(selected_manifest)
    print("Sweep state:", manifest["state"])
    for case in manifest["cases"]:
        print(f"Set {case['index']}: {case['physical']}")
        for method, outcome in case["methods"].items():
            print(f"  {method}: {outcome['status']}")
            print("   ", outcome.get("output", outcome.get("error", outcome.get("log"))))

In [ ]:
import importlib
import combined_runs
importlib.reload(combined_runs)
import rs_equilibrium
importlib.reload(rs_equilibrium)

if manifest is not None:
    figures_dir = Path("figures").resolve() / Path(selected_manifest).resolve().parent.name
    figures = combined_runs.plot_comparisons(manifest, figures_dir=figures_dir)
